In [5]:
import numpy as np 
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

In [6]:
df = pd.read_csv("../data/raw/crime_incidents_messy.csv")
df.head()

,incident_id,crime_type,district,city,state,address,latitude,longitude,incident_datetime,officer_id,officer_first_name,officer_last_name,badge_number,suspect_id,suspect_first_name,suspect_last_name,suspect_age,suspect_gender,suspect_race,victim_id,victim_first_name,victim_last_name,victim_age,victim_gender,victim_phone,weapon_used,severity,case_status,resolution,num_arrests,property_loss_usd,reported_online,notes
0,INC001115,asslt,Sou,Maplewood,GA,2830 Cedar Lane,170.284100,-77.500710,2024-04-16 08:45:03,OFF0078,Emily,Davis,1342.0,SUS00281,Barbara,Thomas,51.0,MALE,asian,VIC00642,Robert,Torres,51.0,Unknown,6223265920,Firearm,2,Open,No Arrest,NaN,NaN,True,Incident at 2830 Cedar Lane. Officer responded at scene.
1,INC004706,burglary,southeast,Maplewood,OH,2361 Park Rd,29.422717,-77.167016,2022-01-11 22:03:29,OFF0059,Michelle,Green,5133.0,SUS00376,Barbara,Harris,-28.0,Other,Black,VIC00262,John,Martin,231.0,NaN,241-973-4826,Firearm,3,NaN,NaN,2.0,44839.49,yes,Incident at 2361 Park Rd. Officer responded at scene.
2,INC002249,Homocide,Southwest,Lakewood,AZ,1067 Main St,39.411460,-98.615298,2022-04-14 11:39:24,OFF0088,Michael,White,7781.0,SUS00560,Richard,Scott,19.0,F,asian,VIC00518,Richard,Hill,29.0,Other,244-584-7696,KNIFE,1,CLOSED,warning,2.0,15963.38,YES,Incident at 1067 Main St. Officer responded at scene.
3,INC000021,Property Damage,Cen,Springfield,AZ,4713 Washington Ave,28.633439,-99.030051,2020-12-09 15:14:24,OFF0077,Chris,Jackson,5097.0,SUS00468,Michael,Thomas,262.0,MALE,Black,VIC00243,Susan,Young,243.0,NaN,5021227484,hands,Low,Resolved,NaN,5.0,48680.14,False,Incident at 4713 Washington Ave. Officer responded at scene.
4,INC000488,Domestc Violence,North,Lakewood,PA,1371 River Rd,34.217850,-121.614731,2021-07-24 20:08:13,OFF0083,Susan,Flores,6220.0,SUS00751,Thomas,Jackson,75.0,female,white,VIC00512,Thomas,Thomas,51.0,M,9698766873,Unarmed,MEDIUM,Closed,Warning Issued,2.0,23513.01,YES,NaN


In [7]:
df.shape

(5250, 33)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         5250 non-null   str    
 1   crime_type          5250 non-null   str    
 2   district            5250 non-null   str    
 3   city                5250 non-null   str    
 4   state               5250 non-null   str    
 5   address             5250 non-null   str    
 6   latitude            4992 non-null   float64
 7   longitude           4961 non-null   float64
 8   incident_datetime   4910 non-null   str    
 9   officer_id          5250 non-null   str    
 10  officer_first_name  5250 non-null   str    
 11  officer_last_name   5250 non-null   str    
 12  badge_number        4938 non-null   float64
 13  suspect_id          4440 non-null   str    
 14  suspect_first_name  4440 non-null   str    
 15  suspect_last_name   4440 non-null   str    
 16  suspect_age      

In [9]:
def clean_property_loss_usd(val):
    if pd.isna(val):
        return np.nan
    val_str = val.strip()

    if val_str.endswith(".0") and val_str[:-2].count('.') >= 1:
        val_str = val_str[:-2]

    try:
        return float(val_str)
    except ValueError:
        return np.nan

    

In [10]:
def load_and_clean_data(filepath:str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    initial_count = len(df)
    df.property_loss_usd = df.property_loss_usd.apply(clean_property_loss_usd)
    df = df.dropna(subset=["property_loss_usd"])
    dropped_count = initial_count - len(df)
    print(f"Cleaned property loss. Dropped {dropped_count} rows with unrecoverable targets.")
    df.property_loss_usd = df.property_loss_usd.apply(lambda x: max(0.0,x))
    
    return df
    

In [11]:
df = load_and_clean_data(filepath="../data/raw/crime_incidents_messy.csv")

Cleaned property loss. Dropped 436 rows with unrecoverable targets.


In [12]:
df.head()

,incident_id,crime_type,district,city,state,address,latitude,longitude,incident_datetime,officer_id,officer_first_name,officer_last_name,badge_number,suspect_id,suspect_first_name,suspect_last_name,suspect_age,suspect_gender,suspect_race,victim_id,victim_first_name,victim_last_name,victim_age,victim_gender,victim_phone,weapon_used,severity,case_status,resolution,num_arrests,property_loss_usd,reported_online,notes
1,INC004706,burglary,southeast,Maplewood,OH,2361 Park Rd,29.422717,-77.167016,2022-01-11 22:03:29,OFF0059,Michelle,Green,5133.0,SUS00376,Barbara,Harris,-28.0,Other,Black,VIC00262,John,Martin,231.0,NaN,241-973-4826,Firearm,3,NaN,NaN,2.0,44839.49,yes,Incident at 2361 Park Rd. Officer responded at scene.
2,INC002249,Homocide,Southwest,Lakewood,AZ,1067 Main St,39.411460,-98.615298,2022-04-14 11:39:24,OFF0088,Michael,White,7781.0,SUS00560,Richard,Scott,19.0,F,asian,VIC00518,Richard,Hill,29.0,Other,244-584-7696,KNIFE,1,CLOSED,warning,2.0,15963.38,YES,Incident at 1067 Main St. Officer responded at scene.
3,INC000021,Property Damage,Cen,Springfield,AZ,4713 Washington Ave,28.633439,-99.030051,2020-12-09 15:14:24,OFF0077,Chris,Jackson,5097.0,SUS00468,Michael,Thomas,262.0,MALE,Black,VIC00243,Susan,Young,243.0,NaN,5021227484,hands,Low,Resolved,NaN,5.0,48680.14,False,Incident at 4713 Washington Ave. Officer responded at scene.
4,INC000488,Domestc Violence,North,Lakewood,PA,1371 River Rd,34.217850,-121.614731,2021-07-24 20:08:13,OFF0083,Susan,Flores,6220.0,SUS00751,Thomas,Jackson,75.0,female,white,VIC00512,Thomas,Thomas,51.0,M,9698766873,Unarmed,MEDIUM,Closed,Warning Issued,2.0,23513.01,YES,NaN
7,INC003981,manslaughter,midtown,Centerville,IL,1610 River Rd,46.854678,-99.512808,2023-09-30 14:49:01,OFF0008,Karen,Hall,7869.0,SUS00533,Jennifer,Taylor,73.0,MALE,White,VIC00904,David,Torres,NaN,female,587-505-1798,NaN,1,open,No Arrest,2.0,19075.36,no,Incident at 1610 River Rd. Officer responded at scene.


In [15]:
import re

def clean_crime_type(val):
    if pd.isna(val):
        return "Unknown"
    
    # 1. Normalize spacing and title case
    text = str(val).strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse double/triple spaces
    text = text.title()
    
    # 2. Standardization mapping dictionary for typos, abbreviations, and synonyms
    mapping = {
        # Arson
        "Arsen": "Arson",
        "Fire Setting": "Arson",
        
        # Assault & Battery variants
        "Asslt": "Assault",
        "Assault  &  Battery": "Assault & Battery",
        
        # Breaking & Entering / Burglary
        "B&E": "Breaking & Entering",
        "Burglry": "Burglary",
        
        # Cyber Crime / Online Fraud / Hacking
        "Cybercrime": "Cyber Crime",
        "Hacking": "Cyber Crime",
        "Online Fraud": "Cyber Crime",
        
        # Domestic Violence
        "Dv": "Domestic Violence",
        "Dom. Violence": "Domestic Violence",
        "Domestc Violence": "Domestic Violence",
        "Domestic  Violence": "Domestic Violence",
        
        # Drug Offense / Narcotics
        "Drug  Offence": "Drug Offense",
        "Drug  Offense": "Drug Offense", # (or Drug Offence)
        "Drug Offence": "Drug Offense",
        "Drugs": "Drug Offense",
        "Narcotics": "Drug Offense",
        
        # DUI / Drunk Driving / DWI
        "D.U.I.": "Dui",
        "Duii": "Dui",
        "Drunk Driving": "Dui",
        "Dwi": "Dui",
        
        # Fraud / Scam / Deception
        "Fraudulent Activity": "Fraud",
        
        # Homicide / Murder / Manslaughter
        "Homocide": "Homicide",
        
        # Kidnapping
        "Kidnaping": "Kidnapping",
        
        # Property Damage / Vandalism / Graffiti
        "Graffiti": "Vandalism",
        "Vandlism": "Vandalism",
        
        # Robbery
        "Robbry": "Robbery",
        "Roberry": "Robbery",
        "Armed  Robbery": "Armed Robbery",
        
        # Sexual Assault
        "Sa": "Sexual Assault",
        "Sex Assault": "Sexual Assault",
        "Sex  Assault": "Sexual Assault",
        "Sexual  Assault": "Sexual Assault",
        "Sexual  Assualt": "Sexual Assault",
        "Sexual Assualt": "Sexual Assault",
        
        # Theft / Larceny / Stealing
        "Stealing": "Theft",
        "Larceny": "Theft",
        "Theft/Larceny": "Theft",
        
        # Trespassing
        "Trespass": "Trespassing",
        "Tresspassing": "Trespassing"
    }
    
    return mapping.get(text, text)

In [14]:
df.crime_type = df.crime_type.apply(

crime_type
Abduction               59
Armed  Robbery           2
Armed Robbery           37
Arsen                   69
Arson                  184
Assault                150
Assault  &  Battery      1
Assault & Battery       57
Asslt                   42
B&E                     45
Battery                 47
Breaking & Entering     42
Burglary               159
Burglry                 56
Cyber Crime            149
Cybercrime              69
D.U.I.                  58
Deception               74
Dom. Violence           73
Domestc Violence        52
Domestic  Violence       3
Domestic Violence       97
Drug  Offence            1
Drug  Offense            2
Drug Offence            75
Drug Offense           146
Drugs                   54
Drunk Driving           65
Dui                    105
Duii                    64
Dv                      83
Dwi                     51
Fire Setting            79
Fraud                  172
Fraudulent Activity     66
Graffiti                58
Hacking          